# Smart Obstacle Narrator — training
This notebook trains **your annotated eight-class dataset**. No custom data or experimental results are bundled.
Select a Colab GPU runtime. Put the delivered ZIP at `MyDrive/SmartObstacleNarrator/source/SmartObstacleNarrator.zip`.
Prepare the dataset locally with `son-data prepare`; ZIP its contents (images/, labels/, manifest.csv at ZIP root), then upload as `datasets/dataset_v1.zip`.
Only load trusted PyTorch weights. The notebook saves experiments and checkpoints directly to Drive; local dataset extraction is disposable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, sys, subprocess, os
DRIVE = Path('/content/drive/MyDrive/SmartObstacleNarrator

In [ ]:
')
ARCHIVE = DRIVE / 'source' / 'SmartObstacleNarrator.zip'
assert ARCHIVE.is_file(), f'Upload the delivered project ZIP to {ARCHIVE}'
PROJECT_PARENT = Path('/content/son_source')
PROJECT_PARENT.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    for item in z.infolist():
        if not (PROJECT_PARENT/item.filename).resolve().is_relative_to(PROJECT_PARENT.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(PROJECT_PARENT)
PROJECT = PROJECT_PARENT / 'SmartObstacleNarrator'
os.chdir(PROJECT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'])
sys.path.insert(0, str(PROJECT/'src'))
for name in ('datasets','models','experiments','results','notebooks','documentation'):
    (DRIVE/name).mkdir(parents=True, exist_ok=True)

## Verify compute and stage the data
Do not change the local dataset path between an interrupted run and its resume.

In [ ]:
import torch, shutil, json, yaml
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
DATA_ARCHIVE = DRIVE / 'datasets' / 'dataset_v1.zip'
DATASET = Path('/content/son_dataset/v1')
assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
if DATASET.exists():
    shutil.rmtree(DATASET)  # This exact temporary staging directory only.
DATASET.mkdir(parents=True)
with zipfile.ZipFile(DATA_ARCHIVE) as z:
    for item in z.infolist():
        if not (DATASET/item.filename).resolve().is_relative_to(DATASET.resolve()):
            raise ValueError('Unsafe dataset archive')
    z.extractall(DATASET)
assert (DATASET/'manifest.csv').is_file(), 'ZIP must have manifest.csv at its root, not an extra parent directory.'

## Validate before allocating training time
Review class counts and flagged near-duplicates. Fix leakage before proceeding. Empty label files are required for negative images.

In [ ]:
from smart_obstacle_narrator.dataset import validate
REPORT = validate(DATASET)
print(json.dumps(REPORT, indent=2))
assert REPORT['valid'], REPORT['errors']
(DRIVE/'results'/'dataset_v1_validation.json').write_text(json.dumps(REPORT, indent=2))

## Configure the experiment
Start with a short smoke run, then a new run ID for full training. `INTERRUPT_AFTER=1` deliberately stops after one saved epoch to test recovery; next session set it to None and RESUME=True. Keep the run ID, version, data and configuration unchanged. Set NEAR_DUPLICATES_REVIEWED only after inspecting/removing flagged leakage.

In [ ]:
RUN_ID = 'yolo11n_v1_run01'
VERSION = 'v1'
RESUME = False
INTERRUPT_AFTER = None
NEAR_DUPLICATES_REVIEWED = False
CONFIG = yaml.safe_load((PROJECT/'configs'/'training.yaml').read_text())
print(CONFIG)

## Train, evaluate, and publish an immutable model bundle
`last.pt`, `best.pt`, and epoch checkpoints go to Drive during training. An interrupted current epoch can be lost. A disconnect during writing can corrupt the newest checkpoint: preserve periodic checkpoints. Validation selects the model; test results are for reporting, not repeated tuning. The release includes metrics, plots, predictions, hashes and environment versions.

In [ ]:
from smart_obstacle_narrator.training import train_release
metadata = train_release(DATASET, DRIVE, RUN_ID, VERSION, CONFIG,
                         resume=RESUME, device='0',
                         near_duplicates_reviewed=NEAR_DUPLICATES_REVIEWED,
                         interrupt_after=INTERRUPT_AFTER)
print(json.dumps(metadata, indent=2))

## Inspect metrics and plots
Results are actual outputs, not project targets. Inspect per-class failures. Recall of stairs/poles matters even if aggregate mAP looks good.

In [ ]:
from IPython.display import display, Image
for split in ('val','test'):
    print(split, metadata['metrics'][split])
    for plot in (DRIVE/'results'/VERSION/split).glob('*.png'):
        print(plot.name)
        display(Image(filename=str(plot), width=850))
for plot in (DRIVE/'experiments'/RUN_ID).glob('*.png'):
    display(Image(filename=str(plot), width=850))

## Verify persistent release
Download the entire model version directory to local `models/v1/`. Keep the Drive copy. The separate inference notebook needs no training variables.

In [ ]:
from smart_obstacle_narrator.release import verify_release
weights = verify_release(DRIVE/'models'/VERSION)
print('Verified persistent model:', weights)
print('Checkpoint recovery information:', DRIVE/'experiments'/RUN_ID/'checkpoint_status.json')